# MELD preparation for SECNet
Run from the project or meld_processed directory. The first code cell binds the single CONFIG object used by all modules. All outputs use actual downloaded MELD. Human listening remains pending.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file() and (p / "src/meld_preprocess").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from config import CONFIG
CONFIG["OUTPUT_DIR"] = 'meld_processed'
CONFIG  # the canonical object; edit config.py to change defaults

{'SAMPLE_RATE': 16000,
 'N_MELS': 80,
 'N_FFT': 512,
 'WIN_LENGTH': 400,
 'HOP_LENGTH': 160,
 'MAX_FRAMES': 1000,
 'MAX_TOKENS': 50,
 'MIN_FREQ': 2,
 'EMB_DIM': 300,
 'SEED': 42,
 'CENTER': True,
 'PAD_MODE': 'constant',
 'WINDOW': 'hann',
 'POWER': 2.0,
 'MEL_HTK': False,
 'MEL_NORM': 'slaney',
 'FMIN': 0.0,
 'FMAX': 8000.0,
 'LOG_SCALE': 'natural',
 'EPS': 1e-10,
 'STD_EPS': 1e-08,
 'NUM_WORKERS': 4,
 'DOWNLOAD_RETRIES': 5,
 'HTTP_TIMEOUT': 90,
 'FFMPEG_TIMEOUT': 120,
 'FFMPEG_RETRIES': 2,
 'CHUNK_BYTES': 1048576,
 'DISK_RESERVE_BYTES': 2147483648,
 'OUTPUT_DIR': 'meld_processed',
 'TEMP_DIR': 'work',
 'RAW_DIR': 'raw',
 'OOV_STD': 0.1,
 'NORM_ATOL': 1e-05,
 'REPRO_ATOL': 1e-06,
 'CSV_COMMIT': 'e8cedf27b5d2877e198332c957127e16eb214afe',
 'MELD_SHA256': 'a56b4407d574195cbce470d86f9c9d72fcfea59b0e34502ecd4babee4a5c613e',
 'MELD_BYTES': 10878146150,
 'MELD_URL': 'https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz',
 'GLOVE_URL': 'https://huggingface.co/stanfor

In [2]:
from meld_preprocess.pipeline import Pipeline
pipeline = Pipeline(ROOT, CONFIG)

## 1. Official sources and integrity

In [3]:
pipeline.stage("download", resume=True)

START download


PASS download: 24.76s


## 3–9, 12. CSV, labels, speakers, shifts and train vocabulary

In [4]:
pipeline.stage("text", resume=True)

START text


C:\Users\OS\Downloads\học hành kì này\HocSau\meld_project\src\meld_preprocess\pipeline.py:96: UserWarning: SOURCE DISCREPANCY: official train CSV has 1038 dialogues; ID 60 absent. See MELD issue #25. No rows altered.
  read_source(self.raw / "csv" / f"{split}_sent_emo.csv", split), split


PASS text: 1.62s


## 10. GloVe embedding

In [5]:
pipeline.stage("embedding", resume=True)

START embedding


PASS embedding: 5.01s


## 1–2. Safe nested extraction and mono WAV

In [6]:
pipeline.stage("audio", resume=True)

START audio


C:\Users\OS\Downloads\học hành kì này\HocSau\meld_project\src\meld_preprocess\pipeline.py:96: UserWarning: SOURCE DISCREPANCY: official train CSV has 1038 dialogues; ID 60 absent. See MELD issue #25. No rows altered.
  read_source(self.raw / "csv" / f"{split}_sent_emo.csv", split), split


WAV conversion:   0%|          | 0/5 [00:00<?, ?it/s]

WAV conversion: 100%|██████████| 5/5 [00:00<00:00, 240.05it/s]

WAV conversion:   0%|          | 0/9989 [00:00<?, ?it/s]

WAV conversion:  84%|████████▍ | 8374/9989 [00:10<00:01, 837.15it/s]

... (6 outputs left out)

WAV conversion: 100%|██████████| 5/5 [00:00<00:00, 913.31it/s]

WAV conversion:   0%|          | 0/2610 [00:00<?, ?it/s]

WAV conversion: 100%|██████████| 2610/2610 [00:02<00:00, 1040.79it/s]

PASS audio: 257.96s


## 11. Log-mel and train-only normalization

In [7]:
pipeline.stage("features", resume=True)

START features


Mel train:   0%|          | 0/9989 [00:00<?, ?it/s]

C:\Users\OS\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mel train:  24%|██▍       | 2376/9989 [00:10<00:32, 237.52it/s]

Mel train:  59%|█████▉    | 5887/9989 [00:20<00:13, 304.27it/s]

Mel train:  94%|█████████▍| 9398/9989 [00:30<00:01, 322.18it/s]

... (2 outputs left out)

Mel dev: 100%|██████████| 1109/1109 [00:02<00:00, 421.49it/s]

Mel test:   0%|          | 0/2610 [00:00<?, ?it/s]

Mel test: 100%|██████████| 2610/2610 [00:06<00:00, 385.08it/s]

PASS features: 55.35s


## 13–14. CSV/PKL/PT/JSON/NPY and Dataset

In [8]:
pipeline.stage("package", resume=True)

START package


C:\Users\OS\Downloads\học hành kì này\HocSau\meld_project\src\meld_preprocess\pipeline.py:96: UserWarning: SOURCE DISCREPANCY: official train CSV has 1038 dialogues; ID 60 absent. See MELD issue #25. No rows altered.
  read_source(self.raw / "csv" / f"{split}_sent_emo.csv", split), split


PASS package: 2.84s


## Full integrity and actual DataLoader
Validation runs in a script so Windows spawn workers have a safe main entry point.

In [9]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "scripts/validate_data.py"), "--all"], cwd=ROOT, check=True)

CompletedProcess(args=['C:\\Users\\OS\\Downloads\\học hành kì này\\HocSau\\meld_project\\.venv\\Scripts\\python.exe', 'C:\\Users\\OS\\Downloads\\học hành kì này\\HocSau\\meld_project\\scripts\\validate_data.py', '--all'], returncode=0)

In [10]:
from dataset import MELDDataset, collate_fn
from torch.utils.data import DataLoader
dataset = MELDDataset(split="train", root=pipeline.out)
batch = next(iter(DataLoader(dataset, batch_size=4, collate_fn=collate_fn)))
for key, value in batch.items():
    print(key, value.shape, value.dtype)

tokens torch.Size([4, 14, 32]) torch.int64
token_mask torch.Size([4, 14, 32]) torch.bool
audio torch.Size([4, 14, 830, 80]) torch.float32
audio_mask torch.Size([4, 14, 830]) torch.bool
has_audio torch.Size([4, 14]) torch.bool
speaker torch.Size([4, 14]) torch.int64
utt_mask torch.Size([4, 14]) torch.bool
emotion torch.Size([4, 14]) torch.int64
sentiment torch.Size([4, 14]) torch.int64
shift torch.Size([4, 14]) torch.int64


## Reproducibility on every real valid WAV

In [11]:
from meld_preprocess.validation import reproduce
reproduce(ROOT)

Reproduce train:   0%|          | 0/9989 [00:00<?, ?it/s]

Reproduce train:  35%|███▌      | 3513/9989 [00:10<00:18, 351.28it/s]

Reproduce train:  70%|███████   | 7026/9989 [00:21<00:09, 327.88it/s]

Reproduce train: 100%|██████████| 9989/9989 [00:30<00:00, 325.18it/s]

Reproduce dev:   0%|          | 0/1109 [00:00<?, ?it/s]

Reproduce dev: 100%|██████████| 1109/1109 [00:03<00:00, 352.57it/s]

Reproduce test:   0%|          | 0/2610 [00:00<?, ?it/s]

Reproduce test: 100%|██████████| 2610/2610 [00:07<00:00, 338.96it/s]

{'status': 'PASS',
 'audio_recomputed': 13706,
 'audio_max_abs_error': 0.0,
 'tolerance': 1e-06,
 'embedding_array_equal': True,
 'train_statistics_recomputed': True,
 'serialization': 'Logical tensor/array equality verified. No claim of bitwise-identical PT container serialization.',
 'seconds': 48.1866071999998}

In [12]:
from meld_preprocess.reporting import reports
reports(ROOT)

## Manual listening — PENDING_MANUAL_REVIEW
Listen and compare the original text. The notebook does not automatically confirm human review.

In [13]:
import json
from IPython.display import Audio, display
for item in json.loads((pipeline.out / "audio_review.json").read_text(encoding="utf-8")):
    print(item["split"], item["dialogue_id"], item["utt_id"], item["text_original"], item["status"])
    display(Audio(filename=str(ROOT / item["wav"])))

dev 47 10 Sooo!! Anyway... PENDING_MANUAL_REVIEW


train 191 5 She’s sleeping. PENDING_MANUAL_REVIEW


train 39 15 Ah, it’s says so in the script! PENDING_MANUAL_REVIEW


test 117 2 Thanks. PENDING_MANUAL_REVIEW


train 482 0 Oh my God! How did you get back here?! PENDING_MANUAL_REVIEW
